In [156]:
from IPython.terminal.shortcuts import next_history_or_next_completion
from zmq.eventloop import future

from connect import cursor
import pandas as pd
from prophet import Prophet

Pesquisa no Banco de dados, teste de conexao

In [5]:


cursor.execute("select * from cafeteria.coffe c")

fetchall - Retornando pesquisa do banco de dados

In [6]:
row = cursor.fetchall()

In [7]:
df = pd.DataFrame(row)

In [8]:
df.head()


,0,1,2,3,4,5,6,7,8,9,10
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


Removendo Nulos Dropna

In [9]:
df = df.dropna()

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3547 entries, 0 to 3546
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       3547 non-null   int64  
 1   1       3547 non-null   str    
 2   2       3547 non-null   float64
 3   3       3547 non-null   str    
 4   4       3547 non-null   str    
 5   5       3547 non-null   str    
 6   6       3547 non-null   str    
 7   7       3547 non-null   int64  
 8   8       3547 non-null   int64  
 9   9       3547 non-null   str    
 10  10      3547 non-null   str    
dtypes: float64(1), int64(3), str(7)
memory usage: 304.9 KB


In [11]:
df[9] = pd.to_datetime(df[9])

In [12]:
df.head()

,0,1,2,3,4,5,6,7,8,9,10
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


In [13]:
media_2025 = df[(df[9] >= '2025-01-01') & (df[9] <= '2025-12-31')]


In [57]:
media_2025

,0,1,2,3,4,5,6,7,8,9,10
2580,18,card,35.76,Cocoa,Night,Tue,Mar,2,3,2025-03-18,18:42:30.606000
2605,9,card,30.86,Americano with Milk,Morning,Thu,Jan,4,1,2025-01-02,09:58:05.335000
2606,13,card,35.76,Cocoa,Afternoon,Thu,Jan,4,1,2025-01-02,13:29:40.128000
2607,13,card,35.76,Cocoa,Afternoon,Thu,Jan,4,1,2025-01-02,13:30:38.720000
2608,14,card,25.96,Cortado,Afternoon,Thu,Jan,4,1,2025-01-02,14:41:10.594000
...,...,...,...,...,...,...,...,...,...,...,...
3542,10,card,35.76,Cappuccino,Morning,Sun,Mar,7,3,2025-03-23,10:34:54.894000
3543,14,card,35.76,Cocoa,Afternoon,Sun,Mar,7,3,2025-03-23,14:43:37.362000
3544,14,card,35.76,Cocoa,Afternoon,Sun,Mar,7,3,2025-03-23,14:44:16.864000
3545,15,card,25.96,Americano,Afternoon,Sun,Mar,7,3,2025-03-23,15:47:28.723000


Vamos Calcular o Lucro anual Bruto de 2025

In [32]:
Bruto_2025 = media_2025[2].sum()

In [33]:
Bruto_2025

np.float64(29600.78)

Bruto anual 2025

Mais pedido em 2025

In [17]:
pedidos = media_2025[3].value_counts().rename_axis('pedido').reset_index(name='quantidade')

In [18]:
pedidos

,pedido,quantidade
0,Americano,244
1,Americano with Milk,189
2,Latte,152
3,Cappuccino,122
4,Cocoa,100
5,Hot Chocolate,70
6,Cortado,34
7,Espresso,32


Pedidos por Periodos


In [19]:
periodoPedidos = (
    media_2025
    .groupby([3, 4])
    .size()
    .reset_index(name='quantidade')
    .sort_values(by=['quantidade'], ascending=False)
)

In [20]:
periodoPedidos

,3,4,quantidade
1,Americano,Morning,103
0,Americano,Afternoon,102
4,Americano with Milk,Morning,92
21,Latte,Afternoon,77
6,Cappuccino,Afternoon,54
23,Latte,Night,53
5,Americano with Milk,Night,50
3,Americano with Milk,Afternoon,47
11,Cocoa,Night,47
2,Americano,Night,39


Melhores meses

In [21]:
mesesVen = (
    media_2025
    .groupby([3, 6])
    .size()
    .reset_index(name='MesVendas')
    .sort_values(by=['MesVendas'], ascending=False)
)

In [22]:
mesesVen

,3,6,MesVendas
0,Americano,Feb,117
2,Americano,Mar,102
3,Americano with Milk,Feb,85
23,Latte,Mar,57
9,Cocoa,Feb,56
21,Latte,Feb,54
6,Cappuccino,Feb,52
4,Americano with Milk,Jan,52
5,Americano with Milk,Mar,52
8,Cappuccino,Mar,43


Salvando pesquisas

In [23]:
with pd.ExcelWriter('Analise2025.xlsx') as writer:
    media_2025.to_excel(writer, sheet_name='2025', index=False)
    pedidos.to_excel(writer, sheet_name='Pedidos', index=False)
    periodoPedidos.to_excel(writer, sheet_name='Periodo', index=False)
    mesesVen.to_excel(writer, sheet_name='MesVendas', index=False)


Previsao de Venda 2025 - restante do ano

In [168]:
vendas_mes2025 = (
    media_2025
    .groupby(media_2025.columns[6])[media_2025.columns[2]].sum()
    .reset_index()
    .rename(columns={media_2025.columns[6]: 'MesVendas',
                     media_2025.columns[2]: 'Quantidade'})
    .sort_values(by=['Quantidade', 'MesVendas'])


)

In [169]:
vendas_mes2025

,MesVendas,Quantidade
1,Jan,6398.86
2,Mar,9986.44
0,Feb,13215.48


In [215]:
previsao = media_2025[[2,9]]

In [216]:
previsao = previsao.rename(columns={2: 'y',
                        9 :'ds'})

In [217]:
previsao

,y,ds
2580,35.76,2025-03-18
2605,30.86,2025-01-02
2606,35.76,2025-01-02
2607,35.76,2025-01-02
2608,25.96,2025-01-02
...,...,...
3542,35.76,2025-03-23
3543,35.76,2025-03-23
3544,35.76,2025-03-23
3545,25.96,2025-03-23


In [218]:
modelo = Prophet()
modelo.fit(previsao)

01:27:48 - cmdstanpy - INFO - Chain [1] start processing
01:27:48 - cmdstanpy - INFO - Chain [1] done processing


In [219]:
m = Prophet()
m.fit(previsao)

01:27:49 - cmdstanpy - INFO - Chain [1] start processing
01:27:49 - cmdstanpy - INFO - Chain [1] done processing


In [220]:
future = m.make_future_dataframe(periods=365)
future.tail()

,ds
439,2026-03-19
440,2026-03-20
441,2026-03-21
442,2026-03-22
443,2026-03-23


In [221]:
forecast = m.predict(future)
forecast[['ds', 'yhat']].tail()


,ds,yhat
439,2026-03-19,27.813428
440,2026-03-20,28.184906
441,2026-03-21,27.611751
442,2026-03-22,28.692340
443,2026-03-23,28.520755
